# 12 Failure Modes & Production Debugging: Loop Detection Guard

**Scenario**: Implement an automated loop detection guard class that monitors tool calls and halts execution to prevent resource waste.

## Step 1: Implement Loop Detection Guard Class

In [1]:
import json

class LoopDetectionGuard:
    def __init__(self, max_steps=5):
        self.max_steps = max_steps
        self.action_history = []
        
    def record_and_verify(self, action_name, action_args):
        # Normalize arguments to string for hashable lookup
        action_signature = (action_name, json.dumps(action_args, sort_keys=True))
        
        # Check if action signature already exists in history (indicating a loop)
        if action_signature in self.action_history:
            print(f"\n[LOOP GUARD ALERT] Duplicate action signature detected: {action_signature}")
            print("Execution halted to prevent infinite billing loop!")
            return False
            
        self.action_history.append(action_signature)
        
        if len(self.action_history) > self.max_steps:
            print(f"\n[LOOP GUARD ALERT] Maximum steps threshold ({self.max_steps}) exceeded.")
            return False
            
        return True

## Step 2: Simulate Loop Failures and Guard Halt Trigger

In [2]:
class MockLoopingAgent:
    def __init__(self):
        self.guard = LoopDetectionGuard(max_steps=4)
        
    def run_agent(self, mode="looping"):
        print(f"\nStarting Mock Agent in '{mode}' mode...")
        step = 0
        
        while step < 10:
            step += 1
            
            # Define actions
            if mode == "looping":
                # Simulates repeating the same database query because of unparsed output
                action_name = "QueryDatabase"
                action_args = {"query": "SELECT * FROM users;"}
            else:
                # Correct execution progression
                action_name = f"QueryDatabaseStep{step}"
                action_args = {"query": f"SELECT * FROM users LIMIT {step};"}
                
            print(f"  Step {step}: Proposing [{action_name}] with args: {action_args}")
            
            # Verify with loop detection guard before executing tool
            is_safe = self.guard.record_and_verify(action_name, action_args)
            if not is_safe:
                print("[STATUS] Loop guard triggered halt. Safely aborted.")
                return "ABORTED"
                
            print(f"  Step {step}: Tool executed successfully.")
            
        return "COMPLETED"

agent = MockLoopingAgent()
agent.run_agent(mode="looping")

print("-" * 55)

agent_healthy = MockLoopingAgent()
agent_healthy.run_agent(mode="healthy")


Starting Mock Agent in 'looping' mode...
  Step 1: Proposing [QueryDatabase] with args: {'query': 'SELECT * FROM users;'}
  Step 1: Tool executed successfully.
  Step 2: Proposing [QueryDatabase] with args: {'query': 'SELECT * FROM users;'}

[LOOP GUARD ALERT] Duplicate action signature detected: ('QueryDatabase', '{"query": "SELECT * FROM users;"}')
Execution halted to prevent infinite billing loop!
[STATUS] Loop guard triggered halt. Safely aborted.
-------------------------------------------------------

Starting Mock Agent in 'healthy' mode...
  Step 1: Proposing [QueryDatabaseStep1] with args: {'query': 'SELECT * FROM users LIMIT 1;'}
  Step 1: Tool executed successfully.
  Step 2: Proposing [QueryDatabaseStep2] with args: {'query': 'SELECT * FROM users LIMIT 2;'}
  Step 2: Tool executed successfully.
  Step 3: Proposing [QueryDatabaseStep3] with args: {'query': 'SELECT * FROM users LIMIT 3;'}
  Step 3: Tool executed successfully.
  Step 4: Proposing [QueryDatabaseStep4] with arg

'ABORTED'

## Output Explanation & Complexity Analysis

- **Loop Detection Halt**: The simulator catches the duplicate `QueryDatabase` trace, immediately flags a `LOOP GUARD ALERT`, and triggers a clean halt to prevent infinite recursion.
- **Time Complexity**: $O(T_{\text{history}})$ time checking signatures.
- **Space Complexity**: $O(T_{\text{history}})$ storage memory requirements.
- **Component Denotations**:
  - $T_{\text{history}}$: Number of prior steps logged and checked in active memory.